# P-Question Pipeline — Colab Runner

In [ ]:
# ── Block 1: Git pull ─────────────────────────────────────────────────────────
import os

REPO = 'computational_semantics_course'

if os.path.isdir(f'/content/{REPO}'):
    %cd /content/{REPO}
    !git pull
else:
    %cd /content
    !git clone https://github.com/yuvalzohar12/computational_semantics_course.git
    %cd /content/{REPO}

# Clear stale bytecode so pulled changes take effect immediately
!find . -name '*.pyc' -delete 2>/dev/null
!find . -name '__pycache__' -type d -exec rm -rf {} + 2>/dev/null

!pip install -q -r requirements.txt
!python -m spacy download en_core_web_sm -q
print('Done.')

In [ ]:
# ── Block 2: Data load ────────────────────────────────────────────────────────
from google.colab import drive
import os, shutil

drive.mount('/content/drive')

# ↓ Update this path if the dataset is stored elsewhere in your Drive
DRIVE_DATASET_PATH = '/content/drive/MyDrive/ConTRoL-dataset'

os.makedirs('data', exist_ok=True)
if os.path.exists(DRIVE_DATASET_PATH):
    shutil.copytree(DRIVE_DATASET_PATH, 'data/ConTRoL-dataset', dirs_exist_ok=True)
    print('Dataset copied successfully.')
else:
    print(f'Dataset not found at: {DRIVE_DATASET_PATH}')
    print('Update DRIVE_DATASET_PATH to the correct location in your Drive.')

In [ ]:
# ── Block 3: Model choice and API key ────────────────────────────────────────
import os

# ── Pick ONE model and fill in the matching API key ───────────────────────────
#
#  Provider   Model name                Key to fill in
#  --------   ----------                --------------
#  Groq       llama-3.1-8b-instant      GROQ_API_KEY
#  Groq       llama-3.3-70b-versatile   GROQ_API_KEY
#  HuggingFace qwen2.5-32b-instruct     HF_API_KEY

MODEL = 'qwen2.5-32b-instruct'          # ← change here

os.environ['GROQ_API_KEY'] = ''                          # ← Groq key
os.environ['HF_API_KEY']   = 'YOUR_HF_API_KEY_HERE'     # ← HuggingFace key

print(f'Model : {MODEL}')
print(f'GROQ  : {"set" if os.environ["GROQ_API_KEY"] else "empty"}')
print(f'HF    : {"set" if os.environ["HF_API_KEY"] else "empty"}')

In [ ]:
# ── Block 4: Run experiment ───────────────────────────────────────────────────

LIMIT      = 10     # number of samples — set to None for the full dataset
MAX_TOKENS = 2048   # max tokens per LLM call

limit_flag = f'--limit {LIMIT}' if LIMIT else ''

!python main.py --experiment p_question --model {MODEL} --max_tokens {MAX_TOKENS} {limit_flag}

In [ ]:
# ── Block 6: Save to Drive & see results ─────────────────────────────────────
import json, glob, shutil, os

SEP = '=' * 72

DRIVE_RESULTS_DIR = '/content/drive/MyDrive/p_question_results'
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)

result_files = sorted(glob.glob('results/p_question*.json'))
if not result_files:
    print('No results yet — run Block 4 first.')
else:
    # ── Save latest run to Drive ──────────────────────────────────────────────
    latest_local = result_files[-1]
    dest = os.path.join(DRIVE_RESULTS_DIR, os.path.basename(latest_local))
    shutil.copy2(latest_local, dest)
    print(f'Saved to Drive: {dest}\n')

    # ── All runs summary table ────────────────────────────────────────────────
    all_files = sorted(glob.glob(f'{DRIVE_RESULTS_DIR}/p_question*.json'))
    print(SEP)
    print(f'All saved runs ({len(all_files)} total):')
    print(SEP)
    print(f"  {'Timestamp':<20} {'Model':<30} {'N':>5} {'Acc':>6}  {'Ent':>5}  {'Con':>5}  {'Neu':>5}")
    print(f"  {'-'*75}")
    for fpath in all_files:
        try:
            with open(fpath) as f:
                d = json.load(f)
            meta  = d['metadata']
            samps = d['samples']
            n     = len(samps)
            correct = sum(1 for s in samps if s['label'] == s['prediction'])
            acc   = f'{correct/n:.1%}' if n else 'n/a'
            ts    = meta.get('timestamp', '')[:19]
            model = meta.get('model', '?')[:28]
            def label_acc(lbl):
                g = [s for s in samps if s['label'] == lbl]
                h = sum(1 for s in g if s['prediction'] == lbl)
                return f'{h/len(g):.0%}' if g else 'n/a'
            marker = '  ← new' if fpath == dest else ''
            print(f"  {ts:<20} {model:<30} {n:>5} {acc:>6}  "
                  f"{label_acc('Entailment'):>5}  {label_acc('Contradiction'):>5}  "
                  f"{label_acc('Neutral'):>5}{marker}")
        except Exception as e:
            print(f"  [error reading {os.path.basename(fpath)}: {e}]")
    print(SEP)

    # ── Detailed results for the new run ──────────────────────────────────────
    with open(latest_local) as f:
        data = json.load(f)

    samples = data['samples']
    total   = len(samples)
    correct = sum(1 for s in samples if s['label'] == s['prediction'])

    print()
    print(SEP)
    print(f"NEW RUN — {data['metadata']['model']}   Accuracy: {correct}/{total} = {correct/total:.1%}" if total else 'No samples.')
    print(SEP)
    print(f"  {'Label':<15} {'Correct':>7} {'Total':>7} {'Accuracy':>9}")
    print(f"  {'-'*42}")
    for lbl in ['Entailment', 'Contradiction', 'Neutral']:
        gold = [s for s in samples if s['label'] == lbl]
        hits = sum(1 for s in gold if s['prediction'] == lbl)
        pct  = f'{hits/len(gold):.1%}' if gold else 'n/a'
        print(f"  {lbl:<15} {hits:>7} {len(gold):>7} {pct:>9}")

    for s in samples:
        verdict = '✓' if s['label'] == s['prediction'] else '✗'

        print()
        print(SEP)
        print(f"[{verdict}] ID: {s['id']}   Gold: {s['label']}   Pred: {s['prediction']}")
        print(f"Premise    : {s['premise'][:200].strip()}...")
        print(f"Hypothesis : {s['hypothesis']}")

        # Stage 1a — all questions with source tag
        sources = s.get('stage1a_sources', ['LLM'] * len(s['stage1a_questions']))
        ner_count = sources.count('NER')
        print()
        print(f"  Stage 1a — {len(s['stage1a_questions'])} questions  ({ner_count} NER, {len(sources)-ner_count} LLM)")
        for i, (q, src) in enumerate(zip(s['stage1a_questions'], sources), 1):
            print(f"    {i:>2}. [{src}] {q}")

        # Stage 1b — top-K with scores and source
        print()
        print(f"  Stage 1b — top-{len(s['stage1b_aligned'])} by alignment score")
        print(f"    {'Score':>8}   {'Src':<5}  Question")
        print(f"    {'-'*65}")
        for item in s['stage1b_aligned']:
            src = item.get('source', 'LLM')
            print(f"    {item['score']:>8.4f}   [{src}]  {item['question']}")

        # Stage 1c — evidence + answer per question
        print()
        print(f"  Stage 1c — answers")
        for i, item in enumerate(s['stage1c_answers'], 1):
            src    = item.get('source', 'LLM')
            status = 'UNANSWERABLE' if item['unanswerable'] else f"score={item['score']:.4f}"
            print(f"    Q{i}. [{src}] [{status}] {item['question']}")
            evidence = item.get('evidence_sentences', [])
            if evidence:
                print(f"        Evidence ({len(evidence)} sentence{'s' if len(evidence) != 1 else ''}):")
                for ev in evidence:
                    print(f"          - {ev}")
            else:
                print(f"          Evidence : (none gathered)")
            print(f"        Answer   : {item['answer']}")

        # Stage 2
        print()
        print(f"  Stage 2 — classification")
        print(f"    Evidence : {s['stage2_evidence'] or '(empty — all unanswerable)'}")
        print(f"    Mode     : {s['stage2_mode']}   Metric: {s['alignment_metric']}")
        print(f"    Gold     : {s['label']}   Pred: {s['prediction']}   {verdict}")
        if s.get('warnings'):
            print(f"    Warnings : {'; '.join(s['warnings'])}")

    print()
    print(SEP)